# Módulo de visualização e insights para identificação de desertos médicos Sus

**Autora:** Vanessa Batista ([@vandromedae](https://github.com/vandromedae))  
**Repositório:** [github.com/vandromedae/desertos-medicos-sus](https://github.com/vandromedae/desertos-medicos-sus)  
**Licença:** [MIT](https://github.com/vandromedae/desertos-medicos-sus/blob/main/LICENSE)

---

In [ ]:
# ============================================================
#  Desertos Médicos SUS - Visualização e Insights
# ============================================================


import sys
from pathlib import Path

# Adicionar raiz do projeto ao path
notebook_path = Path().resolve()
project_root = notebook_path.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
import folium
from folium import plugins

from src.config import (
    DATA_PROCESSED, DATA_EXTERNAL, OUTPUT_MAPAS, OUTPUT_RELATORIOS,
)
from src.visualization import (
    mapa_densidade_municipal,
    mapa_bivariado_municipal,
    mapa_densidade_setorial_por_municipio,
    mapa_bivariado_setorial_por_municipio,
)

# Configurações de visualização
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.3f' % x)

# Garantir diretórios de saída
OUTPUT_MAPAS.mkdir(parents=True, exist_ok=True)
OUTPUT_RELATORIOS.mkdir(parents=True, exist_ok=True)

print(" Ambiente configurado!")
print(f" Raiz do projeto: {project_root}")
print(f" Mapas de saída: {OUTPUT_MAPAS}")

In [ ]:
# ============================================================
# Carregar todos os dados processados
# ============================================================

print("=" * 70)
print("CARREGANDO DADOS PROCESSADOS")
print("=" * 70)

# 1. Base municipal
arquivo_municipal = DATA_PROCESSED / "base_municipal_para_mapa.parquet"
if not arquivo_municipal.exists():
    raise FileNotFoundError(f"Arquivo não encontrado: {arquivo_municipal}")
df_municipal = pd.read_parquet(arquivo_municipal)
print(f" Base municipal: {len(df_municipal):,} municípios")

# 2. Setores com acessibilidade (com polígonos reais)
arquivo_setores = DATA_PROCESSED / "setores_com_acessibilidade_real.parquet"
if not arquivo_setores.exists():
    raise FileNotFoundError(f"Arquivo não encontrado: {arquivo_setores}")
df_setores = pd.read_parquet(arquivo_setores)
print(f" Setores com acessibilidade: {len(df_setores):,} setores")

# 3. CNES agregados
arquivo_cnes = DATA_PROCESSED / "cnes_agregados.parquet"
if not arquivo_cnes.exists():
    raise FileNotFoundError(f"Arquivo não encontrado: {arquivo_cnes}")
df_cnes = pd.read_parquet(arquivo_cnes)
print(f" CNES agregados: {len(df_cnes):,} estabelecimentos")

# 4. Comparação municipal vs acessibilidade
arquivo_comparacao = DATA_PROCESSED / "comparacao_municipal_vs_e2sfca.parquet"
if arquivo_comparacao.exists():
    df_comparacao = pd.read_parquet(arquivo_comparacao)
    print(f" Comparação municipal vs acessibilidade: {len(df_comparacao):,} municípios")
else:
    df_comparacao = None
    print(" Arquivo de comparação não encontrado")

# 5. Shapefile de municípios de SP
shapefile_municipios = DATA_EXTERNAL / "SP_Municipios_2025" / "SP_Municipios_2025.shp"
if shapefile_municipios.exists():
    gdf_municipios = gpd.read_file(shapefile_municipios)
    gdf_municipios['cod_mun_6'] = gdf_municipios['CD_MUN'].astype(str).str[:6]
    print(f" Shapefile de municípios: {len(gdf_municipios):,} polígonos")
else:
    gdf_municipios = None
    print(" Shapefile de municípios não encontrado - mapa coroplético indisponível")

# 6. Shapefile de setores censitários
shapefile_setores = DATA_EXTERNAL / "SP_setores_CD2022_IBGE" / "SP_setores_CD2022.shp"
if shapefile_setores.exists():
    gdf_setores_geo = gpd.read_file(shapefile_setores)
    print(f" Shapefile de setores: {len(gdf_setores_geo):,} polígonos")
else:
    gdf_setores_geo = None
    print(" Shapefile de setores não encontrado")

print(f"\n Resumo dos dados:")
print(f"   Municípios: {len(df_municipal):,}")
print(f"   Setores censitários: {len(df_setores):,}")
print(f"   CNES: {len(df_cnes):,}")
print(f"   População total: {df_municipal['populacao'].sum():,}")
print(f"   Médicos SUS: {df_municipal['total_medicos'].sum():,}")

In [ ]:
# ============================================================
# MAPA 1: Densidade Médica por Município (Coroplético)
# ============================================================

print("=" * 70)
print("GERANDO MAPA 1: DENSIDADE MÉDICA MUNICIPAL")
print("=" * 70)

if gdf_municipios is None:
    print(" Shapefile de municípios não disponível - pulando mapa 1")
else:
    output_map = OUTPUT_MAPAS / 'mapa_01_densidade_municipal.html'
    m1 = mapa_densidade_municipal(
        gdf_municipios=gdf_municipios,
        df_municipal=df_municipal,
        output_path=output_map,
    )
    print(f"\n Mapa 1 salvo em: {output_map}")
    display(m1)

In [ ]:
# ============================================================
# MAPA 2: Densidade Populacional - Um mapa por município
# ============================================================

print("=" * 70)
print("GERANDO MAPAS DE DENSIDADE POPULACIONAL POR MUNICÍPIO")
print("=" * 70)

if gdf_setores_geo is None:
    print(" Shapefile de setores não disponível - pulando")
else:
    arquivos_gerados = mapa_densidade_setorial_por_municipio(
        gdf_setores=gdf_setores_geo,
        df_setores=df_setores,
    )
    print(f"\n {len(arquivos_gerados)} mapas gerados!")

In [ ]:
# ============================================================
# MAPA 3: Bivariado Municipal - Densidade Médica × Acessibilidade (E2SFCA)
# ============================================================

print("=" * 70)
print("GERANDO MAPA 3: BIVARIADO MUNICIPAL (E2SFCA)")
print("=" * 70)

if gdf_municipios is None or df_comparacao is None:
    print(" Dados municipais não disponíveis - pulando")
else:
    output_map = OUTPUT_MAPAS / 'mapa_03_bivariado_municipal.html'
    m3 = mapa_bivariado_municipal(
        gdf_municipios=gdf_municipios,
        df_comparacao=df_comparacao,
        output_path=output_map,
    )
    print(f"\n Mapa 3 salvo em: {output_map}")
    display(m3)

In [ ]:
# ============================================================
# MAPA 4: Bivariado Setorial - Densidade Pop × Acesso a Médicos (E2SFCA)
# ============================================================

print("=" * 70)
print("GERANDO MAPA 4: BIVARIADO SETORIAL POR MUNICÍPIO")
print("=" * 70)

if gdf_setores_geo is None:
    print(" Shapefile de setores não disponível - pulando")
else:
    arquivos_gerados = mapa_bivariado_setorial_por_municipio(
        gdf_setores=gdf_setores_geo,
        df_setores=df_setores,
    )
    print(f"\n {len(arquivos_gerados)} mapas bivariados gerados!")